# Ask Spider: Gradio + fine-tuned Qwen

Notebook 1 trained the **3-epoch LoRA adapter**. This app reuses Notebook 2's BGE
schema retrieval, NF4 Qwen + that adapter, and read-only SQLite execution.
The app also displays the saved 100-example benchmark (3% → 15% → 62% EX).

**Run on Kaggle with T4 GPU + Internet.** To interact with the app, edit this
notebook, set `LAUNCH_APP = True` below, and run all cells. Choose a temporary
app password when prompted, then open the printed Gradio URL. Username: `demo`.
Keep the interactive Kaggle session running. The API-submitted verification run
uses `False`, performs three GPU smoke tests, and exits without a public server.


In [ ]:
LAUNCH_APP = False  # Set True in an INTERACTIVE Kaggle session.
import os, sys, subprocess
from pathlib import Path
assert Path('/kaggle/input').is_dir(), 'Run this notebook on Kaggle only'
os.environ['HF_HOME'] = '/tmp/hf-gradio-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
os.environ['GRADIO_TEMP_DIR'] = '/tmp/gradio-app-files'
os.chdir('/kaggle/working')
for package in ('notebook2', 'gradio_app'):
    Path(package).mkdir(exist_ok=True)
    Path(package, '__init__.py').write_text('')
sys.path.insert(0, '/kaggle/working')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *['chromadb==1.5.9', 'fastembed==0.9.1', 'transformers==5.17.0', 'peft==0.21.2', 'tokenizers==0.23.3', 'accelerate>=1.0', 'bitsandbytes==0.50.2', 'safetensors>=0.5', 'sqlglot==30.30.0', 'pandas==2.3.3', 'nvidia-ml-py>=12', 'gradio==6.30.0']], check=True)


## 1. Load the existing pipeline and the Gradio frontend
These modules are bundled; no separate Python file uploads are needed.

In [ ]:
%%writefile /kaggle/working/notebook2/environment.py
"""Chunk 1: persistent Chroma configuration and remote-only verification."""
import importlib.metadata
import json
import logging
import os
from pathlib import Path

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
COLLECTION_NAME = "spider_ddl_bge_small_v1"
VRAM_LIMIT_BYTES = 10_000_000_000  # Decimal GB, stricter than 10 GiB.


def require_kaggle():
    if not Path("/kaggle/input").is_dir() or not Path("/kaggle/working").is_dir():
        raise RuntimeError("Run this notebook on Kaggle with a T4 GPU; local execution is disabled.")


def configure_logging(output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, force=True,
        format="%(asctime)s %(levelname)s %(message)s",
        handlers=[logging.StreamHandler(), logging.FileHandler(output / "evaluation.log")])


def open_collection(directory):
    import chromadb
    from chromadb.config import Settings
    client = chromadb.PersistentClient(path=str(directory),
        settings=Settings(anonymized_telemetry=False))
    collection = client.get_or_create_collection(COLLECTION_NAME, embedding_function=None,
        configuration={"hnsw": {"space": "cosine"}},
        metadata={"embedding_model": EMBEDDING_MODEL, "schema_format": "sqlite-ddl-v1"})
    if collection.metadata.get("embedding_model") != EMBEDDING_MODEL:
        raise ValueError("Existing collection uses a different embedding model")
    return client, collection


def verify_environment():
    """Remote assertion test, before any schema or model work."""
    require_kaggle()
    import sqlite3
    import tempfile
    import torch
    import sqlglot
    assert torch.cuda.is_available(), "Enable the Kaggle GPU accelerator"
    assert "T4" in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
    assert sqlglot.parse_one("SELECT 1", read="sqlite") is not None
    with sqlite3.connect(":memory:") as conn:
        assert conn.execute("SELECT 1").fetchone() == (1,)
    with tempfile.TemporaryDirectory() as directory:
        client, collection = open_collection(directory)
        collection.add(ids=["probe"], embeddings=[[1.0, 0.0]], documents=["SELECT 1"],
                       metadatas=[{"db_id": "probe"}])
        _, reopened = open_collection(directory)
        assert reopened.get(where={"db_id": "probe"})["ids"] == ["probe"]
    versions = {name: importlib.metadata.version(name) for name in
        ("torch", "chromadb", "fastembed", "transformers", "peft", "bitsandbytes", "pandas", "sqlglot")}
    logging.info("CHUNK 1 PASS | GPU=%s | versions=%s", torch.cuda.get_device_name(0), versions)
    return versions


In [ ]:
%%writefile /kaggle/working/notebook2/schemas.py
"""Chunk 2: per-table SQLite DDL, CPU BGE vectors, strict database filtering."""
from contextlib import closing
import hashlib
import logging
from pathlib import Path
import re
import sqlite3

from notebook2.environment import EMBEDDING_MODEL, open_collection


def database_path(root, db_id):
    if not re.fullmatch(r"[A-Za-z0-9_-]+", db_id):
        raise ValueError("Invalid Spider database ID")
    root = Path(root).resolve()
    path = (root / db_id / f"{db_id}.sqlite").resolve()
    if not path.is_relative_to(root) or not path.is_file():
        raise FileNotFoundError(path)
    return path


def extract_schemas(path, db_id):
    """Original CREATE TABLE retains column types, composite PKs and FK clauses.

    PRAGMA metadata supplements DDL and explicitly records key relationships.
    No data values, validation questions, or gold SQL enter the vector index.
    """
    records = []
    with closing(sqlite3.connect(Path(path).resolve().as_uri() + "?mode=ro", uri=True)) as conn:
        tables = conn.execute("SELECT name, sql FROM sqlite_master WHERE type='table' "
                              "AND name NOT LIKE 'sqlite_%' AND sql IS NOT NULL ORDER BY name").fetchall()
        for name, ddl in tables:
            quoted = '"' + name.replace('"', '""') + '"'
            columns = conn.execute(f"PRAGMA table_info({quoted})").fetchall()
            keys = conn.execute(f"PRAGMA foreign_key_list({quoted})").fetchall()
            primary = [r[1] for r in sorted(columns, key=lambda r: r[5]) if r[5]]
            lines = [f"-- Database: {db_id}; table: {name}", ddl.rstrip("; ") + ";"]
            if primary:
                lines.append("-- Primary key: " + ", ".join(primary))
            for fk in keys:
                lines.append(f"-- Foreign key {fk[0]}.{fk[1]}: {name}.{fk[3]} -> {fk[2]}.{fk[4] or '(primary key)'}")
            document = "\n".join(lines)
            records.append({"id": hashlib.sha256(f"{db_id}\0{name}".encode()).hexdigest(),
                "document": document, "metadata": {"db_id": db_id, "table_name": name,
                    "schema_sha256": hashlib.sha256(document.encode()).hexdigest()}})
    if not records:
        raise ValueError(f"No tables in {db_id}")
    return records


class SchemaIndex:
    def __init__(self, directory):
        from fastembed import TextEmbedding
        self.client, self.collection = open_collection(directory)
        # ONNX CPU embeddings leave GPU memory exclusively for the language model.
        self.encoder = TextEmbedding(model_name=EMBEDDING_MODEL,
            cache_dir="/tmp/bge-cache", providers=["CPUExecutionProvider"], threads=2)

    def ingest(self, path, db_id):
        records = extract_schemas(path, db_id)
        vectors = [v.tolist() for v in self.encoder.embed([r["document"] for r in records])]
        assert all(len(v) == 384 for v in vectors)
        self.collection.upsert(ids=[r["id"] for r in records], embeddings=vectors,
            documents=[r["document"] for r in records], metadatas=[r["metadata"] for r in records])
        stale = set(self.collection.get(where={"db_id": db_id})["ids"]) - {r["id"] for r in records}
        if stale:
            self.collection.delete(ids=sorted(stale))
        logging.info("Indexed %s: %d tables", db_id, len(records))
        return records

    def retrieve(self, user_query, db_id, top_k=3):
        if not user_query.strip() or top_k < 1:
            raise ValueError("Non-empty question and positive top_k required")
        available = self.collection.get(where={"db_id": db_id})["ids"]
        if not available:
            raise KeyError(f"Database {db_id!r} has not been indexed")
        vector = next(iter(self.encoder.query_embed(user_query))).tolist()
        result = self.collection.query(query_embeddings=[vector],
            where={"db_id": db_id}, n_results=min(top_k, len(available)),
            include=["documents", "metadatas", "distances"])
        matches = [{"document": doc, "metadata": meta, "distance": distance}
            for doc, meta, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0])]
        assert matches and all(m["metadata"]["db_id"] == db_id for m in matches), "Cross-database retrieval"
        return "\n\n".join(m["document"] for m in matches), matches

    def retrieve_relevant_schemas(self, user_query: str, db_id: str, top_k: int = 3) -> str:
        return self.retrieve(user_query, db_id, top_k)[0]


_active_index = None


def initialize_schema_index(directory):
    global _active_index
    _active_index = SchemaIndex(directory)
    return _active_index


def retrieve_relevant_schemas(user_query: str, db_id: str, top_k: int = 3) -> str:
    if _active_index is None:
        raise RuntimeError("Call initialize_schema_index first")
    return _active_index.retrieve_relevant_schemas(user_query, db_id, top_k)


def verify_schemas():
    """Real BGE + Chroma assertion test on remote synthetic databases."""
    import tempfile
    with tempfile.TemporaryDirectory() as directory:
        index = SchemaIndex(Path(directory) / "chroma")
        for db_id in ("alpha", "beta"):
            path = Path(directory) / f"{db_id}.sqlite"
            with sqlite3.connect(path) as conn:
                conn.executescript("CREATE TABLE parent(id INTEGER PRIMARY KEY, name TEXT);"
                    "CREATE TABLE child(id INTEGER PRIMARY KEY, parent_id INTEGER REFERENCES parent(id));")
            records = index.ingest(path, db_id)
            assert len(records) == 2 and "REFERENCES" in records[0]["document"]
            index.ingest(path, db_id)  # Upserts must be idempotent.
        assert index.collection.count() == 4
        for db_id in ("alpha", "beta"):
            text, matches = index.retrieve("List child and parent names", db_id, 20)
            assert len(matches) == 2 and all(m["metadata"]["db_id"] == db_id for m in matches)
            assert f"Database: {db_id}" in text
        try:
            index.retrieve("parent names", "missing")
            raise AssertionError("Unknown database must not fall back to all databases")
        except KeyError:
            pass
    logging.info("CHUNK 2 PASS | real BGE/Chroma isolation, PK/FK extraction, idempotence")


In [ ]:
%%writefile /kaggle/working/notebook2/model.py
"""Chunk 3: native Qwen chat, NF4 + LoRA, bounded single-example inference."""
import logging
import threading
import time

from notebook2.environment import VRAM_LIMIT_BYTES, require_kaggle

BASE_MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
BASE_REVISION = "4858886f896bd05db823557fc5cbc4ac28342af7"
SYSTEM_PROMPT = ("You are an expert SQL assistant. Given the following database schema, "
    "write a valid SQL query that accurately answers the question.\n\n"
    "### Database Schema:\n{retrieved_schema}")


def chat_messages(question, db_id, retrieved_schema):
    if not retrieved_schema.strip():
        raise ValueError("Refusing schema-free inference in Notebook 2")
    return [{"role": "system", "content": SYSTEM_PROMPT.format(retrieved_schema=retrieved_schema)},
            {"role": "user", "content": f"Database: {db_id}\nQuestion: {question}"}]


def assistant_target(query):
    """Reference formatting only; never append a gold target during inference."""
    return {"role": "assistant", "content": f"```sql\n{query}\n```"}


class MemoryBudgetExceeded(RuntimeError):
    pass


class MemoryMonitor:
    """Sample NVML device usage and record exact PyTorch allocator high-water marks.

    NVML includes context and non-PyTorch allocations. Its peak is sampled at
    50 ms; the allocator is additionally capped at 8.8 GB. This is a measured
    budget check, not a promise about every possible unsampled transient.
    """
    def __init__(self, torch):
        import pynvml
        self.torch, self.nvml = torch, pynvml
        pynvml.nvmlInit()
        uuid = str(torch.cuda.get_device_properties(0).uuid)
        self.handle = pynvml.nvmlDeviceGetHandleByUUID(uuid)
        self.peak_device_bytes = 0
        self.error = None
        self.stop_event = threading.Event()
        self.thread = threading.Thread(target=self._watch, daemon=True)
        self.thread.start()

    def _watch(self):
        try:
            while not self.stop_event.is_set():
                self.sample()
                self.stop_event.wait(0.05)
        except Exception as exc:
            self.error = str(exc)

    def sample(self):
        used = self.nvml.nvmlDeviceGetMemoryInfo(self.handle).used
        self.peak_device_bytes = max(self.peak_device_bytes, used)

    def snapshot(self):
        self.sample()
        return {"peak_allocated_bytes": self.torch.cuda.max_memory_allocated(0),
            "peak_reserved_bytes": self.torch.cuda.max_memory_reserved(0),
            "peak_device_used_bytes_sampled": self.peak_device_bytes,
            "limit_bytes": VRAM_LIMIT_BYTES, "nvml_sampling_seconds": 0.05}

    def check(self):
        if self.error:
            raise RuntimeError("NVML monitoring failed: " + self.error)
        stats = self.snapshot()
        if max(stats[k] for k in ("peak_reserved_bytes", "peak_device_used_bytes_sampled")) >= VRAM_LIMIT_BYTES:
            raise MemoryBudgetExceeded(f"10 GB VRAM budget exceeded: {stats}")
        return stats

    def close(self):
        self.stop_event.set()
        self.thread.join(timeout=2)


class SQLGenerator:
    def __init__(self, adapter="./lora_sql_adapter_3epochs", max_context=2048, max_new_tokens=256):
        require_kaggle()
        import torch
        from peft import PeftConfig, PeftModel
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
        assert torch.cuda.is_available()
        self.torch = torch
        self.max_context, self.max_new_tokens = max_context, max_new_tokens
        set_seed(3407)
        torch.cuda.set_per_process_memory_fraction(8_800_000_000 / torch.cuda.get_device_properties(0).total_memory, 0)
        torch.cuda.reset_peak_memory_stats(0)
        self.memory = MemoryMonitor(torch)
        config = PeftConfig.from_pretrained(adapter)
        assert config.r == 16 and config.lora_alpha == 16 and config.bias == "none"
        assert set(config.target_modules) == {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
        self.tokenizer = AutoTokenizer.from_pretrained(adapter)
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=BASE_REVISION,
            device_map={"": "cuda:0"}, torch_dtype=torch.float16,
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16))
        self.model = PeftModel.from_pretrained(base, adapter, is_trainable=False).eval()
        assert self.model.is_loaded_in_4bit
        quant = self.model.config.quantization_config
        assert (quant.get("bnb_4bit_quant_type") if isinstance(quant, dict) else quant.bnb_4bit_quant_type) == "nf4"
        logging.info("Loaded NF4 base + final LoRA | memory=%s", self.memory.check())

    def __call__(self, question, db_id, schema):
        inputs = self.tokenizer.apply_chat_template(chat_messages(question, db_id, schema),
            tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        prompt_tokens = inputs["input_ids"].shape[1]
        if prompt_tokens + self.max_new_tokens > self.max_context:
            raise ValueError(f"Context limit: {prompt_tokens}+{self.max_new_tokens}>{self.max_context}; no silent truncation")
        inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
        start = time.monotonic()
        with self.torch.no_grad():
            output = self.model.generate(**inputs, do_sample=False, temperature=0.0,
                num_beams=1, max_new_tokens=self.max_new_tokens, use_cache=True,
                pad_token_id=self.tokenizer.eos_token_id)
        tokens = output[0, prompt_tokens:]
        answer = self.tokenizer.decode(tokens, skip_special_tokens=True)
        stats = {"seconds": time.monotonic() - start, "prompt_tokens": prompt_tokens,
            "generated_tokens": len(tokens), "hit_token_limit": len(tokens) == self.max_new_tokens}
        del inputs, output, tokens
        stats["memory"] = self.memory.check()
        return answer, stats


def verify_model(generator):
    """Actual GPU inference and native-template assertions before evaluation."""
    schema = "CREATE TABLE item(id INTEGER PRIMARY KEY);"
    messages = chat_messages("How many items?", "probe", schema)
    assert len(messages) == 2 and messages[0]["content"].endswith(schema)
    assert assistant_target("SELECT 1")["content"] == "```sql\nSELECT 1\n```"
    rendered = generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    assert "<|im_start|>system" in rendered and "<|im_start|>assistant" in rendered
    answer, metadata = generator("How many items?", "probe", schema)
    assert answer.strip() and metadata["generated_tokens"] > 0
    generator.memory.check()
    logging.info("CHUNK 3 PASS | GPU generation=%r | memory=%s", answer, metadata["memory"])


In [ ]:
%%writefile /kaggle/working/notebook2/sql_engine.py
"""Chunk 4: lossless comment stripping and bounded, read-only SQLite scoring."""
from contextlib import closing
import logging
from pathlib import Path
import re
import sqlite3
import time

import pandas as pd
import sqlglot
from sqlglot import exp


def strip_comments(sql):
    """Remove comments outside quoted literals/identifiers, preserving SQL spelling.

    Re-rendering an AST can change result column labels (COUNT(*) vs count(*)),
    so keep the original query for strict Pandas equality with Notebook 1.
    """
    out, i, quote = [], 0, None
    while i < len(sql):
        char = sql[i]
        if quote:
            out.append(char)
            if char == quote:
                if i + 1 < len(sql) and sql[i + 1] == quote and quote != "]":
                    out.append(sql[i + 1])
                    i += 1
                else:
                    quote = None
        elif char in "'\"`[":
            quote = "]" if char == "[" else char
            out.append(char)
        elif sql[i:i+2] == "--":
            end = sql.find("\n", i + 2)
            i = len(sql) if end == -1 else end
            out.append("\n")
            continue
        elif sql[i:i+2] == "/*":
            end = sql.find("*/", i + 2)
            if end == -1:
                raise ValueError("Unterminated SQL comment")
            i = end + 2
            out.append(" ")
            continue
        else:
            out.append(char)
        i += 1
    if quote:
        raise ValueError("Unterminated SQL quote")
    return "".join(out).strip()


def parse_readonly(sql):
    statements = [s for s in sqlglot.parse(sql, read="sqlite") if s is not None]
    if len(statements) != 1 or not isinstance(statements[0], exp.Query):
        raise ValueError("Expected exactly one SELECT/CTE query")
    # Block modifying CTEs as well as top-level mutations.
    if any(isinstance(node, (exp.Insert, exp.Update, exp.Delete, exp.Create, exp.Drop))
           for node in statements[0].walk()):
        raise ValueError("Modifying statements are forbidden")
    return statements[0]


def sanitize_sql(answer):
    blocks = re.findall(r"```(?:sql|sqlite)?\s*\n?(.*?)```", answer, re.I | re.S)
    if len(blocks) > 1 or ("```" in answer and not blocks):
        raise ValueError("Expected one complete SQL code block")
    sql = strip_comments(blocks[0] if blocks else answer)
    parse_readonly(sql)
    return sql


def execute_dataframe(path, sql, timeout=5.0, max_rows=100000):
    if timeout <= 0 or max_rows <= 0:
        raise ValueError("Positive timeout and row limit required")
    parse_readonly(sql)
    allowed = {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ,
               sqlite3.SQLITE_FUNCTION, sqlite3.SQLITE_RECURSIVE}
    def authorize(action, arg1, arg2, db, source):
        if action == sqlite3.SQLITE_FUNCTION and (arg2 or "").lower() == "load_extension":
            return sqlite3.SQLITE_DENY
        return sqlite3.SQLITE_OK if action in allowed else sqlite3.SQLITE_DENY
    start = time.monotonic()
    with closing(sqlite3.connect(Path(path).resolve().as_uri() + "?mode=ro", uri=True, timeout=timeout)) as conn:
        conn.execute("PRAGMA query_only=ON")
        conn.set_authorizer(authorize)
        conn.set_progress_handler(lambda: int(time.monotonic() - start > timeout), 1000)
        cursor = conn.execute(sql)
        rows = cursor.fetchmany(max_rows + 1)
        if len(rows) > max_rows:
            raise ValueError("Result row limit exceeded")
        return pd.DataFrame.from_records(rows, columns=[column[0] for column in cursor.description])


def score_prediction(path, answer, gold):
    result = {"sql": "", "syntax_valid": False, "execution_valid": False,
        "execution_match": False, "error_type": None, "error": None}
    try:
        result["sql"] = sanitize_sql(answer)
    except (ValueError, sqlglot.errors.SqlglotError) as exc:
        return {**result, "error_type": "invalid_sql", "error": str(exc)}
    result["syntax_valid"] = True
    try:
        predicted = execute_dataframe(path, result["sql"])
        result.update(execution_valid=True, execution_match=gold is not None and predicted.equals(gold),
                      result_shape=list(predicted.shape))
    except sqlite3.Error as exc:
        syntax_error = any(marker in str(exc).lower() for marker in
                           ("syntax error", "incomplete input", "unrecognized token"))
        result.update(syntax_valid=not syntax_error,
            error_type="sqlite_syntax" if syntax_error else "sqlite_execution", error=str(exc))
    except (ValueError, sqlglot.errors.SqlglotError) as exc:
        result.update(error_type="resource_or_query_limit", error=str(exc))
    return result


def verify_sql_engine():
    import tempfile
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "fixture.sqlite"
        with sqlite3.connect(path) as conn:
            conn.executescript("CREATE TABLE item(id INTEGER PRIMARY KEY, name TEXT);"
                               "INSERT INTO item VALUES (1,'a'),(2,'b');")
        gold = execute_dataframe(path, "SELECT count(*) FROM item")
        assert score_prediction(path, "```sql\nSELECT count(*) FROM item; -- done\n```", gold)["execution_match"]
        assert not score_prediction(path, "SELECT count(*) AS n FROM item", gold)["execution_match"]
        assert sanitize_sql("SELECT '--literal/*x*/' AS x -- note") == "SELECT '--literal/*x*/' AS x"
        assert not score_prediction(path, "SELECT FROM", gold)["syntax_valid"]
        missing = score_prediction(path, "SELECT * FROM nonexistent", gold)
        assert missing["syntax_valid"] and not missing["execution_valid"]
        for sql in ("DELETE FROM item", "SELECT 1; DROP TABLE item", "ATTACH DATABASE 'x' AS x",
                    "SELECT load_extension('x')", "SELECT 1 /* unclosed"):
            assert not score_prediction(path, sql, gold)["execution_valid"]
        assert not score_prediction(path, "```sql\nSELECT 1\n```\n```sql\nSELECT 2\n```", gold)["syntax_valid"]
        try:
            execute_dataframe(path, "SELECT * FROM item", max_rows=1)
            raise AssertionError("Row cap not enforced")
        except ValueError:
            pass
        try:
            execute_dataframe(path, "WITH RECURSIVE x(n) AS (SELECT 1 UNION ALL SELECT n+1 FROM x) SELECT sum(n) FROM x", timeout=.01)
            raise AssertionError("Timeout not enforced")
        except sqlite3.OperationalError as exc:
            assert "interrupted" in str(exc)
        assert execute_dataframe(path, "SELECT count(*) FROM item").equals(gold)
    logging.info("CHUNK 4 PASS | sanitation, strict equality, syntax/errors, read-only, timeout, row cap")


In [ ]:
%%writefile /kaggle/working/gradio_app/app.py
"""Gradio interface; GPU inference and verification run on Kaggle only."""
import getpass
import logging
from pathlib import Path
import sqlite3
import threading
import time

import gradio as gr
import pandas as pd
import sqlglot

from notebook2.model import MemoryBudgetExceeded
from notebook2.schemas import database_path
from notebook2.sql_engine import sanitize_sql, execute_dataframe

EXAMPLES = [
    ["concert_singer", "How many singers are there?", 3],
    ["concert_singer", "What is the year that had the most concerts?", 3],
    ["concert_singer", "Show the name and theme for all concerts and the number of singers in each concert.", 3],
]


class SQLApp:
    """One shared model; serialize index/model access across all browser sessions."""
    def __init__(self, generator, schema_index, db_root):
        self.generator, self.index = generator, schema_index
        self.db_root = Path(db_root)
        self.databases = sorted(p.parent.name for p in self.db_root.glob("*/*.sqlite")
                                if p.name == p.parent.name + ".sqlite")
        if not self.databases:
            raise ValueError("No Spider databases attached")
        self.indexed = set()
        self.lock = threading.Lock()
        self.unavailable = False

    def ask(self, db_id, question, top_k=3):
        """Return status, SQL, schema, result preview and diagnostic details."""
        sql, schema = "", ""
        empty = pd.DataFrame()
        started = time.monotonic()
        try:
            if db_id not in self.databases:
                raise ValueError("Choose a database from the dropdown")
            if not isinstance(question, str) or not question.strip():
                raise ValueError("Enter a question first")
            if len(question) > 2000:
                raise ValueError("Keep the question under 2,000 characters")
            if top_k not in (1, 2, 3, 4, 5):
                raise ValueError("Choose between 1 and 5 schema tables")
            with self.lock:
                if self.unavailable:
                    raise RuntimeError("GPU limit reached earlier. Restart the Kaggle session and reload the app.")
                path = database_path(self.db_root, db_id)
                if db_id not in self.indexed:
                    self.index.ingest(path, db_id)
                    self.indexed.add(db_id)
                schema, matches = self.index.retrieve(question.strip(), db_id, int(top_k))
                assert all(m["metadata"]["db_id"] == db_id for m in matches)
                raw, metadata = self.generator(question.strip(), db_id, schema)
                sql = sanitize_sql(raw)
                frame = execute_dataframe(path, sql, timeout=5.0, max_rows=20000)
            details = {"database": db_id, "retrieved_tables": [m["metadata"]["table_name"] for m in matches],
                "retrieval_distances": [m.get("distance") for m in matches],
                "total_rows": len(frame), "displayed_rows": min(len(frame), 1000),
                "seconds": round(time.monotonic() - started, 2), "generation": metadata,
                "correctness": "Not evaluated: a new question has no reference answer."}
            status = f"Executed successfully · {len(frame):,} rows · {details['seconds']:.2f}s."
            if len(frame) > 1000:
                status += " Showing the first 1,000 rows."
            status += " Check the SQL and result: successful execution does not guarantee a correct answer."
            logging.info("App request db=%s tables=%s rows=%d", db_id, details["retrieved_tables"], len(frame))
            return status, sql, schema, frame.head(1000), details
        except (MemoryBudgetExceeded, RuntimeError) as exc:
            if isinstance(exc, MemoryBudgetExceeded) or "out of memory" in str(exc).lower() or "NVML" in str(exc):
                self.unavailable = True
            return f"Request failed: {exc}", sql, schema, empty, {"error_type": type(exc).__name__}
        except (ValueError, KeyError, sqlite3.Error, sqlglot.errors.SqlglotError) as exc:
            return f"Request failed: {exc}", sql, schema, empty, {"error_type": type(exc).__name__}


def build_demo(service, benchmark_root):
    benchmark_root = Path(benchmark_root)
    performance = pd.read_csv(benchmark_root / "performance_summary.csv")
    comparison = pd.read_csv(benchmark_root / "comparison.csv")
    with gr.Blocks(title="Ask Spider · Fine-tuned Qwen + schema RAG") as demo:
        gr.Markdown("# Ask Spider\nTurn a question into SQL and inspect the database answer.\n\n"
            "**Notebook 1 trained the SQL adapter. Notebook 2 retrieves schemas and uses that adapter to generate SQL.**")
        with gr.Tab("Ask the database"):
            with gr.Row():
                database = gr.Dropdown(service.databases,
                    value="concert_singer" if "concert_singer" in service.databases else service.databases[0], label="Database")
                top_k = gr.Slider(1, 5, value=3, step=1, label="Tables to retrieve")
            question = gr.Textbox(label="Question", placeholder="How many singers are there?", lines=2)
            run = gr.Button("Generate SQL and run", variant="primary")
            gr.Examples(EXAMPLES, inputs=[database, question, top_k], label="Try an example")
            status = gr.Textbox(label="Status", interactive=False)
            sql = gr.Code(language="sql", label="Generated SQL", interactive=False)
            result = gr.Dataframe(label="Query result (up to 1,000 rows)", interactive=False, type="pandas")
            with gr.Accordion("Retrieved schema", open=False):
                schema = gr.Code(language="sql", interactive=False)
            with gr.Accordion("Timing and model details", open=False):
                details = gr.JSON()
            run.click(service.ask, inputs=[database, question, top_k],
                outputs=[status, sql, schema, result, details], api_name="ask_sql",
                concurrency_limit=1, concurrency_id="single_gpu")
        with gr.Tab("Benchmark results"):
            gr.Markdown("## Saved 100-example benchmark\n"
                "These are the completed November 9 results, not a new benchmark or a confidence score for a live question.\n\n"
                "EX requires exact Pandas result equality, including column labels, types and row order. "
                "The 60-step adapter was not measured on this split.")
            gr.Dataframe(performance, label="Execution accuracy", interactive=False)
            gr.Dataframe(comparison, label="Questions, SQL and execution matches", interactive=False)
        with gr.Tab("How it works"):
            gr.Markdown("""## Two notebooks, one adapted model
1. **Notebook 1:** trains LoRA adapter weights on Spider question-to-SQL examples.
2. **BGE + Chroma:** embed the question and retrieve schema documents from the selected database.
3. **Base Qwen + Notebook 1 adapter:** read the question and retrieved schema through Qwen's chat template, then generate SQL.
4. **SQLite:** executes the cleaned query in read-only mode. Pandas displays the returned rows.

The adapter is loaded once per session. Asking a question does not retrain the model.
BGE retrieves schemas; Qwen writes SQL. No gold SQL is included in a live question's prompt.

The GPU stays on Kaggle. Keep the interactive session running while using this app.
""")
    return demo.queue(default_concurrency_limit=1, max_size=8)


def launch_app(demo):
    """Call from an interactive Kaggle cell; never embed credentials in source."""
    password = getpass.getpass("Choose a temporary app password (not a Kaggle password or API token): ")
    if len(password) < 8:
        raise ValueError("Use at least 8 characters for the app password")
    return demo.launch(share=True, auth=("demo", password), server_name="0.0.0.0",
                       inline=False, prevent_thread_lock=True, show_error=False)


In [ ]:
%%writefile /kaggle/working/gradio_app/verify.py
"""Remote-only HTTP/API smoke verification of the Gradio app."""
import json
from pathlib import Path
import tempfile

import pandas as pd

from notebook2.environment import require_kaggle
from notebook2.sql_engine import execute_dataframe
from gradio_app.app import SQLApp


def verify_app(service, demo, output):
    require_kaggle()
    from gradio_client import Client
    import requests
    checks = []
    for db, question, top_k in (("../invalid", "test", 3), ("concert_singer", "", 3), ("concert_singer", "test", 99)):
        answer = service.ask(db, question, top_k)
        assert answer[0].startswith("Request failed") and answer[3].empty
    checks.append("invalid database, empty question, invalid retrieval count")

    # Verify unsafe/invalid SQL clears prior results using the real SQLite harness.
    class FixtureIndex:
        def ingest(self, path, db_id):
            pass
        def retrieve(self, question, db_id, top_k):
            return "CREATE TABLE singer(id INTEGER);", [{"metadata": {"db_id": db_id, "table_name": "singer"}}]
    for sql in ("DROP TABLE singer", "SELECT * FROM nonexistent", "SELECT 1; DELETE FROM singer"):
        fixture = SQLApp(lambda *args: (sql, {}), FixtureIndex(), service.db_root)
        answer = fixture.ask("concert_singer", "test", 3)
        assert answer[0].startswith("Request failed") and answer[3].empty
    checks.append("mutations, multiple statements and missing-table failures")

    # Exercise the actual Gradio server and serialized event queue on the T4.
    _, local_url, _ = demo.launch(server_name="127.0.0.1", server_port=7860,
        share=False, prevent_thread_lock=True, quiet=True)
    try:
        response = requests.get(local_url, timeout=30)
        assert response.status_code == 200 and "gradio" in response.text.lower()
        client = Client(local_url, verbose=False)
        cases = [
            ("Simple aggregate", "How many singers are there?", "SELECT count(*) FROM singer"),
            ("GROUP BY", "What is the year that had the most concerts?", "SELECT year FROM concert GROUP BY year ORDER BY count(*) DESC LIMIT 1"),
            ("JOIN + GROUP BY", "Show the name and theme for all concerts and the number of singers in each concert.",
             "SELECT T2.concert_name, T2.theme, count(*) FROM singer_in_concert AS T1 JOIN concert AS T2 ON T1.concert_id = T2.concert_id GROUP BY T2.concert_id"),
        ]
        results = []
        for tier, question, gold_sql in cases:
            status, sql, schema, frame_payload, details = client.predict("concert_singer", question, 3, api_name="/ask_sql")
            assert status.startswith("Executed successfully"), status
            assert "Database: concert_singer" in schema
            assert details["database"] == "concert_singer" and details["total_rows"] >= 0
            # Compare values separately from labels to make this a practical app smoke test.
            # This is not a replacement for the existing strict benchmark metric.
            gold = execute_dataframe(service.db_root / "concert_singer/concert_singer.sqlite", gold_sql)
            predicted = execute_dataframe(service.db_root / "concert_singer/concert_singer.sqlite", sql)
            value_match = predicted.shape == gold.shape and predicted.to_numpy().tolist() == gold.to_numpy().tolist()
            results.append({"tier": tier, "question": question, "sql": sql,
                "reference_sql": gold_sql, "value_match": value_match,
                "strict_dataframe_match": predicted.equals(gold), "details": details})
        assert all(r["value_match"] for r in results), results
        checks.append("HTTP page and Gradio client API: three real GPU questions")
        proof = {"execution_location": "Kaggle", "checks_passed": checks,
            "results": results, "memory": service.generator.memory.check()}
        Path(output).write_text(json.dumps(proof, indent=2) + "\n")
        print("GRADIO APP PASS: page, API, three GPU questions and error handling", flush=True)
        return proof
    finally:
        demo.close()


## 2. Verify inputs and load the trained adapter
No retraining and no full benchmark rerun.

In [ ]:
import hashlib, json
from notebook2.environment import configure_logging, verify_environment
from notebook2.model import SQLGenerator
from notebook2.schemas import initialize_schema_index
from gradio_app.app import SQLApp, build_demo, launch_app
from gradio_app.verify import verify_app

configure_logging('/kaggle/working/app_logs')
versions = verify_environment()
inputs = Path('/kaggle/input')
adapters = list(inputs.rglob('lora_sql_adapter_3epochs/adapter_model.safetensors'))
databases = list(inputs.rglob('concert_singer/concert_singer.sqlite'))
benchmarks = list(inputs.rglob('rag_evaluation/summary.json'))
assert len(adapters) == len(databases) == len(benchmarks) == 1
with adapters[0].open('rb') as stream:
    adapter_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
assert adapter_sha == '961d13a4c49a15b92e1a0801f72e86253f4c4f5f7665786e9e758fd6ad1f5a9a'
history_root = benchmarks[0].parent
history = json.loads(benchmarks[0].read_text())
assert history['metadata']['adapter_sha256'] == adapter_sha
assert history['samples'] == 100 and history['execution_accuracy_pct'] == 62.0
db_root = databases[0].parent.parent
generator = SQLGenerator(str(adapters[0].parent))
index = initialize_schema_index('/kaggle/working/app_chroma_store')
service = SQLApp(generator, index, db_root)
demo = build_demo(service, history_root)
print(f'Loaded the final adapter and {len(service.databases)} available databases.')


## 3. Verify the actual Gradio HTTP/API and three live questions
Tests execute on Kaggle. Historical benchmark files are only displayed.

In [ ]:
proof = verify_app(service, demo, '/kaggle/working/gradio_verification.json')
proof['adapter_sha256'] = adapter_sha
proof['versions'] = versions
proof['source_sha256'] = {
    str(p): hashlib.sha256(p.read_bytes()).hexdigest()
    for package in ('notebook2', 'gradio_app') for p in Path(package).glob('*.py')
    if p.name != '__init__.py'
}
Path('gradio_verification.json').write_text(json.dumps(proof, indent=2))
print(json.dumps(proof, indent=2), flush=True)
# A fresh interface after closing the smoke-test server.
demo = build_demo(service, history_root)


## 4. Launch the interactive app
Set `LAUNCH_APP = True` in the first cell, or run `launch_app(demo)` manually here.
Use a new app password, not a Kaggle token. Username: **demo**.

In [ ]:
if LAUNCH_APP:
    launch_app(demo)
else:
    print('Verification finished. For live use, open an interactive T4 session, set LAUNCH_APP=True, and run all cells.')
    print('After setup, additional questions reuse the same loaded model; no retraining is needed.')
